# Kata 28 — Propagación de Errores Multi-Agente

> Spec: [`specs/028-multi-agent-errors/spec.md`](../../specs/028-multi-agent-errors/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Errores en subagentes deben propagarse al coordinador con contexto
estructurado (`failure_type`, `attempted`, `partial_results`,
`alternatives`). Local recovery primero, propagación con contexto
después.

## 2. Por qué importa

`{"results": []}` cuando hubo timeout enmascara el fallo. Generic
`"unavailable"` priva al coordinador del contexto necesario para
recovery.

## 3. Subagente con local recovery + structured propagation

In [ ]:
import random

# Simulación de tools del subagente con failure modes realistas
def http_search(query, fail_mode=None):
    if fail_mode == "timeout":
        raise TimeoutError(f"search timed out: {query}")
    if fail_mode == "permission":
        raise PermissionError(f"forbidden: {query}")
    if fail_mode == "empty_valid":
        return []  # búsqueda exitosa, cero matches
    return [{"title": f"Result for {query}", "url": "https://example.com"}]

def web_search_subagent(query, fail_mode=None):
    """Subagente: local recovery primero, structured propagation después."""
    try:
        results = http_search(query, fail_mode)
        if not results:
            return {"success": True, "results": [], "empty_valid": True, "query": query}
        return {"success": True, "results": results, "query": query}
    except TimeoutError:
        # Local recovery: intenta con query simplificada
        try:
            simpler = " ".join(query.split()[:3])
            results = http_search(simpler, fail_mode=None)  # reintento sin fail
            return {"success": True, "results": results, "query": simpler, "note": "broadened after timeout"}
        except TimeoutError:
            return {
                "success": False,
                "failure_type": "timeout",
                "attempted_query": query,
                "partial_results": [],
                "suggested_alternatives": [" ".join(query.split()[:3]), query.replace("how", "what")],
            }
    except PermissionError as e:
        return {
            "success": False,
            "failure_type": "permission",
            "retryable": False,
            "explanation": str(e),
        }

print("=== caso success ===")
print(json.dumps(web_search_subagent("AI in creative industries"), indent=2, ensure_ascii=False))

print("\n=== caso empty_valid (cero matches, exitoso) ===")
print(json.dumps(web_search_subagent("xyzqq nonexistent topic", fail_mode="empty_valid"), indent=2, ensure_ascii=False))

print("\n=== caso timeout (recovery local) ===")
# Forzamos timeout en primer intento; el broadened reintento no falla
def http_search_timeout_first(query, fail_mode=None, _state={"called": 0}):
    _state["called"] += 1
    if _state["called"] == 1: raise TimeoutError(f"slow: {query}")
    return [{"title": f"Result for {query}", "url": "https://x.com"}]
http_search.__code__ = http_search_timeout_first.__code__
print(json.dumps(web_search_subagent("very long complex query that times out"), indent=2, ensure_ascii=False))

In [ ]:
# Restablecer http_search original para los siguientes ejemplos
def http_search(query, fail_mode=None):
    if fail_mode == "timeout": raise TimeoutError(f"timeout: {query}")
    if fail_mode == "permission": raise PermissionError(f"forbidden: {query}")
    if fail_mode == "empty_valid": return []
    return [{"title": f"Result for {query}", "url": "https://example.com"}]

# Caso permission: no recuperable, propaga estructurado
def web_search_subagent_permission_demo(query):
    try:
        return {"success": True, "results": http_search(query, fail_mode="permission")}
    except PermissionError as e:
        return {"success": False, "failure_type": "permission", "retryable": False, "explanation": str(e)}

print("=== caso permission (no recuperable) ===")
print(json.dumps(web_search_subagent_permission_demo("admin-only data"), indent=2, ensure_ascii=False))

### 3.1 El coordinador decide con el contexto

In [ ]:
def coordinator_handle(subagent_result, query):
    if subagent_result["success"]:
        if subagent_result.get("empty_valid"):
            return {"action": "annotate_coverage_gap", "topic": query}
        return {"action": "use_results", "data": subagent_result["results"]}
    if subagent_result["failure_type"] == "timeout":
        # Reintenta con alternatives
        for alt in subagent_result.get("suggested_alternatives", []):
            r = web_search_subagent(alt)
            if r["success"]:
                return {"action": "recovered_with_alternative", "alternative": alt, "data": r["results"]}
        return {"action": "annotate_coverage_gap", "reason": "timeout, alternatives also failed"}
    if subagent_result["failure_type"] == "permission":
        return {"action": "escalate_to_human", "reason": subagent_result["explanation"]}
    return {"action": "abort", "reason": "unhandled"}

# Demos
queries_with_modes = [
    ("normal query", None),
    ("rare topic", "empty_valid"),
    ("admin data", "permission"),
]
for q, mode in queries_with_modes:
    sub = web_search_subagent_permission_demo(q) if mode == "permission" else web_search_subagent(q, fail_mode=mode)
    coord = coordinator_handle(sub, q)
    print(f"{q:25} | sub.success={sub['success']:5} | coord.action={coord['action']}")

## 4. Anti-patrón — error como success vacío

In [ ]:
def web_search_subagent_BAD(query):
    try:
        return {"success": True, "results": http_search(query, fail_mode="timeout")}
    except (TimeoutError, PermissionError):
        # ❌ Enmascara el fallo como success vacío
        return {"success": True, "results": []}

bad = web_search_subagent_BAD("anything")
print("subagente malo retorna:", bad)
print("\nProblema: el coordinador asume 'no había info', no 'no pudimos buscar'.")
print("El report final tendrá un coverage gap silencioso que NO se reportará.")

## 5. Argumento de certificación

- **Distinguir** access failure (timeout, permission) de **valid empty
  result** (búsqueda exitosa, cero matches).
- **Local recovery primero**: el subagente intenta antes de propagar.
- **Structured error context**: `failure_type`, `attempted`, `partial`,
  `alternatives` permiten al coordinador decidir.
- **Coverage gap annotation** en el output del synthesis cuando un topic
  no tuvo datos.
- Conexión: Kata 06 (estructura del error single-tool), Kata 20 (gap
  reporting en provenance).

## 6. Auto-evaluación

**1. ¿Cómo distingue el coordinador "no había info" vs "no pudimos buscar"?**

Por la combinación `success` + `empty_valid`. `success: true,
empty_valid: true` = no había. `success: false, failure_type: "timeout"`
= no pudimos.

**2. ¿Quién intenta el primer retry: el subagente o el coordinador?**

El subagente — local recovery primero. Sólo cuando agota localmente,
propaga al coordinador con `suggested_alternatives` para que decida.

**3. ¿Qué información mínima necesita el coordinador para decidir
alternative recovery?**

`failure_type`, `attempted_query`, `partial_results` (si los hay),
`suggested_alternatives`. Sin alternatives el coordinador queda
adivinando.